# 04 — PySpark Assembly Pipeline

Load raw NDVI CSVs, join daily PRISM weather on date, aggregate to weekly
tile means, compute the NDVI anomaly target, pivot to wide format, and join
static terrain + soil features. Saves a Parquet dataset ready for MLlib.

**Join order:**
1. Daily NDVI × daily weather → broadcast join on `date`
2. Weekly aggregation: NDVI mean/std, ppt sum, tmax/tmean/vpdmax means
3. Pivot to one row per (tile_id, year) with weekly feature columns
4. Broadcast join terrain (per tile) + soil (per tile)

**Prerequisites:** Java 17 (`sudo apt install -y openjdk-17-jdk`), python3.10 kernel.

In [ ]:
import os, glob

os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['PYSPARK_PYTHON']        = '/usr/bin/python3.10'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/usr/bin/python3.10'

NDVI_DIR  = '../data/ndvi/tiles_100m2'
TERRAIN   = '../data/terrain_100m2.csv'
SOIL      = '../data/soil_100m2.csv'
WEATHER   = '../data/weather_daily.csv'   # daily PRISM, single station, 2016-2025
OUT_DIR   = '../data/spark/assembled'

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('GrapeExpectations-v2')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '200')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

raw_schema = StructType([
    StructField('tile_id', LongType(),   True),
    StructField('date',    StringType(), True),
    StructField('ndvi',    DoubleType(), True),
])

csv_files = glob.glob(os.path.join(NDVI_DIR, 'ndvi_100m2_*.csv'))
print(f'Found {len(csv_files)} year CSVs')

raw = (spark.read
    .option('header', True)
    .schema(raw_schema)
    .csv(csv_files)
    .withColumn('date', F.to_date('date', 'yyyy-MM-dd'))
    .withColumn('year', F.year('date'))
    .withColumn('week', F.weekofyear('date')))

# ── Join daily weather on date (broadcast — only 3653 rows) ─────────────────
weather_schema = StructType([
    StructField('date',   StringType(), True),
    StructField('ppt',    DoubleType(), True),
    StructField('tmax',   DoubleType(), True),
    StructField('tmean',  DoubleType(), True),
    StructField('tmin',   DoubleType(), True),
    StructField('vpdmax', DoubleType(), True),
    StructField('vpdmin', DoubleType(), True),
])
weather = (spark.read
    .option('header', True)
    .schema(weather_schema)
    .csv(WEATHER)
    .withColumn('date', F.to_date('date', 'yyyy-MM-dd')))

raw = raw.join(F.broadcast(weather), on='date', how='left')

print(f'Raw observations: {raw.count():,}')
raw.show(5)

In [ ]:
from pyspark.sql.window import Window

weekly = (raw
    .filter(F.col('ndvi').isNotNull())
    .groupBy('tile_id', 'year', 'week')
    .agg(
        F.mean('ndvi').alias('ndvi_mean'),
        F.stddev('ndvi').alias('ndvi_std'),
        F.count('ndvi').alias('n_obs'),
        F.sum('ppt').alias('ppt_sum'),
        F.mean('tmax').alias('tmax_mean'),
        F.mean('tmean').alias('tmean_mean'),
        F.mean('tmin').alias('tmin_mean'),
        F.mean('vpdmax').alias('vpdmax_mean'),
        F.mean('vpdmin').alias('vpdmin_mean'),
    ))

print(f'Weekly rows: {weekly.count():,}')
weekly.show(5)

In [ ]:
# ── NDVI anomaly: tile deviation from vineyard mean per week × year ─────────
# This strips vintage effects and isolates the spatial (terroir) signal.

vineyard_mean = (weekly
    .groupBy('year', 'week')
    .agg(F.mean('ndvi_mean').alias('vineyard_ndvi_mean')))

weekly = (weekly
    .join(vineyard_mean, on=['year', 'week'], how='left')
    .withColumn('ndvi_anomaly', F.col('ndvi_mean') - F.col('vineyard_ndvi_mean')))

print('Anomaly computed')
weekly.filter(F.col('week').between(36, 43)).show(5)


In [ ]:
weeks_all   = list(range(1, 52))
feature_wks = list(range(1, 36))
target_wks  = list(range(36, 44))

ndvi_wide = (weekly
    .groupBy('tile_id', 'year')
    .pivot('week', weeks_all)
    .agg(
        F.first('ndvi_mean').alias('ndvi_mean'),
        F.first('ndvi_anomaly').alias('ndvi_anomaly'),
        F.first('ppt_sum').alias('ppt_sum'),
        F.first('tmax_mean').alias('tmax_mean'),
        F.first('tmean_mean').alias('tmean_mean'),
        F.first('vpdmax_mean').alias('vpdmax_mean'),
    ))

print(f'Wide shape: {ndvi_wide.count():,} rows × {len(ndvi_wide.columns)} cols')
ndvi_wide.show(3)

In [ ]:
# ── Feature and target summary columns ─────────────────────────────────────
from pyspark.sql import functions as F

feat_mean_cols    = [f'{w}_ndvi_mean'    for w in feature_wks if f'{w}_ndvi_mean'    in ndvi_wide.columns]
target_anom_cols  = [f'{w}_ndvi_anomaly' for w in target_wks  if f'{w}_ndvi_anomaly' in ndvi_wide.columns]

# Season-average anomaly as primary scalar target
ndvi_wide = ndvi_wide.withColumn(
    'ndvi_anomaly_harvest',
    sum(F.col(c) for c in target_anom_cols) / len(target_anom_cols)
)

# Health score: mean, std, CoV of season NDVI
season_mean_cols = [f'{w}_ndvi_mean' for w in range(28, 44) if f'{w}_ndvi_mean' in ndvi_wide.columns]
ndvi_wide = (ndvi_wide
    .withColumn('ndvi_season_mean', sum(F.col(c) for c in season_mean_cols) / len(season_mean_cols)))

print('Feature summary done')


In [ ]:
# ── Join terrain features (static per tile — broadcast join) ──────────────
import pandas as pd

terrain_pd = pd.read_csv(TERRAIN)
terrain_sp = spark.createDataFrame(terrain_pd).cache()
print(f'Terrain: {terrain_sp.count():,} rows × {len(terrain_sp.columns)} cols')

wide = ndvi_wide.join(F.broadcast(terrain_sp), on='tile_id', how='left')


In [ ]:
soil_pd = pd.read_csv(SOIL)
soil_sp = spark.createDataFrame(soil_pd).cache()
print(f'Soil: {soil_sp.count():,} rows × {len(soil_sp.columns)} cols')

wide = wide.join(F.broadcast(soil_sp), on='tile_id', how='left')

print(f'Assembled: {wide.count():,} rows × {len(wide.columns)} cols')

In [ ]:
# ── Save as Parquet (efficient for subsequent MLlib reads) ──────────────────
wide.write.mode('overwrite').parquet(OUT_DIR)
print(f'Saved to {OUT_DIR}')


In [ ]:
check = spark.read.parquet(OUT_DIR)
print(f'Parquet rows: {check.count():,}')
print('Columns:', check.columns[:20], '...')
check.select(
    'tile_id', 'year', 'ndvi_anomaly_harvest',
    'elev_mean', 'ph1to1h2o_r',
    '20_ppt_sum', '20_tmax_mean', '20_vpdmax_mean',
).show(5)